# Pillar 4: ADMET and structural property screeningRun this non-training property screen in Colab after preparing `inputs/admet_smiles.csv`. It is a descriptor/rule-based calculation, not a trained ADMET ML predictor. Treat outputs as exploratory screening.

In [ ]:
from pathlib import Path
import os, sys, subprocess

# Data preparation and audits run in Colab. Code comes from the GitHub checkout.
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except ImportError:
    pass

DATA = Path("/content/drive/MyDrive/Projects/auditddi")
assert DATA.is_dir(), f"Google Drive project data folder not found: {DATA}"
REPO_VALUE = os.environ.get("AUDITDDI_CODE_ROOT", "").strip()
REPO_CANDIDATES = [Path(REPO_VALUE)] if REPO_VALUE else []
REPO_CANDIDATES += [Path.cwd(), Path("/content/auditddi"), Path("/content/auditddi_v2")]
REPO_CANDIDATES += list(Path.cwd().parents)
REPO = next((p.resolve() for p in REPO_CANDIDATES if (p / "src").is_dir() and (p / "requirements_colab.txt").is_file()), None)
if REPO is None:
    raise FileNotFoundError("GitHub checkout not found. Clone/pull https://github.com/akhilau-git/AuditDDI-project.git and set AUDITDDI_CODE_ROOT if needed.")
RESULTS_BASE = DATA / "results"
os.environ["AUDITDDI_DATA_BASE"] = str(DATA)
os.environ["AUDITDDI_RESULTS_BASE"] = str(RESULTS_BASE)
sys.path.insert(0, str(REPO / "src"))
RUNS = RESULTS_BASE / "pillars" / "admet"
RUNS.mkdir(parents=True, exist_ok=True)
print("Colab preparation environment; code:", REPO)
print("Drive data:", DATA)
print("Pillar outputs:", RUNS)


In [ ]:
%pip install -r {REPO / "requirements_colab.txt"}


In [ ]:
import subprocess

INPUT = DATA / "inputs/admet_smiles.csv"
OUT = RUNS
assert INPUT.is_file(), f"Create input CSV: {INPUT}"
subprocess.run([
    sys.executable,
    str(REPO / "pillars/admet/scripts/run_admet_screen.py"),
    "--input", str(INPUT),
    "--output-dir", str(OUT),
], cwd=REPO, check=True)
